In [0]:
from __future__ import annotations

import json

import pandas as pd
from pyspark.sql import functions as F


SCHEMA = "vse_banka.digi_prodej"
TRAINING_CSV = "/Volumes/vse_banka/digi_prodej/bronze_data/demo/vehicle_training_frame.csv"

FEATURE_TABLE = f"{SCHEMA}.gold_features_vehicle_propensity"
CRM_TABLE = f"{SCHEMA}.silver_crm_clients"
STATE_TABLE = f"{SCHEMA}.gold_state_vehicle_insurance_propensity"
TARGETING_TABLE = f"{SCHEMA}.gold_mart_vehicle_campaign_targeting"
METRICS_TABLE = f"{SCHEMA}.gold_model_vehicle_propensity_metrics"

NUMERIC_COLS = [
    "fuel_spend_30d",
    "fuel_txn_count_30d",
    "fuel_spend_90d",
    "service_txn_count_90d",
    "highway_vignette_flag",
    "insurance_payment_flag",
    "days_since_last_insurance",
    "avg_txn_amount",
    "total_txn_count_90d",
]
CATEGORICAL_COLS = ["age_band"]
TARGET_COL = "target_buy_vehicle_insurance_30d"


def load_training_frame() -> pd.DataFrame:
    return (
        spark.read.option("header", True).option("inferSchema", True).csv(TRAINING_CSV).toPandas()
    )


def load_scoring_frame() -> pd.DataFrame:
    return spark.table(FEATURE_TABLE).toPandas()


def fit_model(train_df: pd.DataFrame):
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import average_precision_score, roc_auc_score
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    train = train_df.loc[train_df["dq_flag"].eq("PASS")].copy()
    train["days_since_last_insurance"] = train["days_since_last_insurance"].fillna(9999).clip(upper=9999)

    X = train[NUMERIC_COLS + CATEGORICAL_COLS].copy()
    y = train[TARGET_COL].astype(int)

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imp", SimpleImputer(strategy="constant", fill_value=0.0)),
                        ("scaler", StandardScaler()),
                    ]
                ),
                NUMERIC_COLS,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imp", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                CATEGORICAL_COLS,
            ),
        ]
    )

    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", LogisticRegression(max_iter=300, class_weight="balanced", random_state=42)),
        ]
    )

    model.fit(X, y)
    train_scores = model.predict_proba(X)[:, 1]
    metrics = {
        "roc_auc": float(roc_auc_score(y, train_scores)),
        "average_precision": float(average_precision_score(y, train_scores)),
        "high_threshold": float(pd.Series(train_scores).quantile(0.90)),
        "medium_threshold": float(pd.Series(train_scores).quantile(0.60)),
        "training_rows": int(len(train)),
        "positive_rate": float(y.mean()),
        "model_version": "logreg_v3_databricks_uc_v5",
    }
    return model, metrics


def build_state_table(scoring_df: pd.DataFrame, model, metrics: dict) -> pd.DataFrame:
    score = scoring_df.copy()
    score["days_since_last_insurance_model"] = score["days_since_last_insurance"].fillna(9999).clip(upper=9999)

    X = score[
        [
            "fuel_spend_30d",
            "fuel_txn_count_30d",
            "fuel_spend_90d",
            "service_txn_count_90d",
            "highway_vignette_flag",
            "insurance_payment_flag",
            "days_since_last_insurance_model",
            "avg_txn_amount",
            "total_txn_count_90d",
            "age_band",
        ]
    ].copy()
    X = X.rename(columns={"days_since_last_insurance_model": "days_since_last_insurance"})

    score["propensity_score"] = model.predict_proba(X)[:, 1]
    score["segment"] = pd.cut(
        score["propensity_score"],
        bins=[-float("inf"), metrics["medium_threshold"], metrics["high_threshold"], float("inf")],
        labels=["LOW", "MEDIUM", "HIGH"],
    ).astype(str)

    state = score[
        [
            "record_id",
            "business_date",
            "region",
            "client_id",
            "snapshot_date",
            "propensity_score",
            "segment",
            "fuel_spend_30d",
            "fuel_txn_count_30d",
            "fuel_spend_90d",
            "service_txn_count_90d",
            "highway_vignette_flag",
            "insurance_payment_flag",
            "days_since_last_insurance",
            "avg_txn_amount",
            "total_txn_count_90d",
            "age_band",
            "dq_flag",
        ]
    ].copy()
    state["days_since_last_insurance"] = pd.array(state["days_since_last_insurance"], dtype="Int32")
    return state


def build_targeting_table(state_df: pd.DataFrame) -> pd.DataFrame:
    crm = spark.table(CRM_TABLE).select(
        "client_id",
        "mobile_app_user",
        "onboarding_mode",
        "consent_any_marketing",
        "consent_push",
        "consent_email",
    ).toPandas()
    out = state_df.merge(crm, on="client_id", how="left")
    for col in ["mobile_app_user", "consent_any_marketing", "consent_push", "consent_email"]:
        out[col] = out[col].fillna(False).astype(bool)

    out["recommended_channel_primary"] = None
    out.loc[out["consent_any_marketing"].eq(True), "recommended_channel_primary"] = "banner_in_ib"
    out.loc[out["segment"].eq("MEDIUM") & out["consent_email"].eq(True), "recommended_channel_primary"] = "email"
    out.loc[out["segment"].eq("MEDIUM") & out["mobile_app_user"].eq(True) & out["consent_push"].eq(True), "recommended_channel_primary"] = "push"
    out.loc[out["segment"].eq("HIGH") & out["consent_email"].eq(True), "recommended_channel_primary"] = "email"
    out.loc[out["segment"].eq("HIGH") & out["mobile_app_user"].eq(True) & out["consent_push"].eq(True), "recommended_channel_primary"] = "push"
    out.loc[out["insurance_payment_flag"].eq(1), "recommended_channel_primary"] = None

    out["recommended_channel"] = None
    out.loc[out["consent_any_marketing"].eq(True), "recommended_channel"] = "banner_in_ib"
    out.loc[out["segment"].eq("MEDIUM") & out["consent_email"].eq(True), "recommended_channel"] = "email + banner_in_ib"
    out.loc[out["segment"].eq("MEDIUM") & out["mobile_app_user"].eq(True) & out["consent_push"].eq(True), "recommended_channel"] = "push + banner_in_ib"
    out.loc[out["segment"].eq("HIGH") & out["consent_email"].eq(True), "recommended_channel"] = "email + banner_in_ib"
    out.loc[
        out["segment"].eq("HIGH")
        & out["mobile_app_user"].eq(True)
        & out["consent_push"].eq(True)
        & out["consent_email"].eq(True),
        "recommended_channel",
    ] = "push + banner_in_ib + email"
    out.loc[
        out["segment"].eq("HIGH")
        & out["mobile_app_user"].eq(True)
        & out["consent_push"].eq(True)
        & out["consent_email"].eq(False),
        "recommended_channel",
    ] = "push + banner_in_ib"
    out.loc[out["insurance_payment_flag"].eq(1), "recommended_channel"] = None

    out["recommended_offer"] = "vehicle_insurance_offer"
    out["next_best_action"] = "contact_client"
    out.loc[out["consent_any_marketing"].eq(False), "next_best_action"] = "no_marketing_consent"
    out.loc[out["insurance_payment_flag"].eq(1), "next_best_action"] = "exclude_existing_policy"
    out = out.sort_values(["propensity_score", "client_id"], ascending=[False, True]).reset_index(drop=True)
    out["contact_priority"] = out.index + 1
    out["score_date"] = out["snapshot_date"]

    targeting = out[
        [
            "client_id",
            "propensity_score",
            "segment",
            "recommended_channel_primary",
            "recommended_channel",
            "recommended_offer",
            "next_best_action",
            "mobile_app_user",
            "onboarding_mode",
            "consent_any_marketing",
            "consent_push",
            "consent_email",
            "contact_priority",
            "score_date",
        ]
    ].copy()
    targeting["contact_priority"] = pd.array(targeting["contact_priority"], dtype="Int32")
    return targeting


def build_segment_summary(state_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        state_df["segment"]
        .value_counts(dropna=False)
        .rename_axis("segment")
        .reset_index(name="client_count")
    )

    segment_order = ["LOW", "MEDIUM", "HIGH"]
    summary["segment"] = pd.Categorical(summary["segment"], categories=segment_order, ordered=True)
    summary = summary.sort_values("segment").reset_index(drop=True)
    summary["segment"] = summary["segment"].astype(str)
    return summary


def persist_tables(state_df: pd.DataFrame, targeting_df: pd.DataFrame, metrics: dict) -> None:
    state_sdf = spark.createDataFrame(state_df).select(
        F.col("record_id").cast("string").alias("record_id"),
        F.col("business_date").cast("date").alias("business_date"),
        F.col("region").cast("string").alias("region"),
        F.col("client_id").cast("string").alias("client_id"),
        F.col("snapshot_date").cast("date").alias("snapshot_date"),
        F.col("propensity_score").cast("double").alias("propensity_score"),
        F.col("segment").cast("string").alias("segment"),
        F.col("fuel_spend_30d").cast("double").alias("fuel_spend_30d"),
        F.col("fuel_txn_count_30d").cast("bigint").alias("fuel_txn_count_30d"),
        F.col("fuel_spend_90d").cast("double").alias("fuel_spend_90d"),
        F.col("service_txn_count_90d").cast("bigint").alias("service_txn_count_90d"),
        F.col("highway_vignette_flag").cast("bigint").alias("highway_vignette_flag"),
        F.col("insurance_payment_flag").cast("bigint").alias("insurance_payment_flag"),
        F.col("days_since_last_insurance").cast("int").alias("days_since_last_insurance"),
        F.col("avg_txn_amount").cast("double").alias("avg_txn_amount"),
        F.col("total_txn_count_90d").cast("bigint").alias("total_txn_count_90d"),
        F.col("age_band").cast("string").alias("age_band"),
        F.col("dq_flag").cast("string").alias("dq_flag"),
    )
    state_sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STATE_TABLE)

    targeting_sdf = spark.createDataFrame(targeting_df).select(
        F.col("client_id").cast("string").alias("client_id"),
        F.col("propensity_score").cast("double").alias("propensity_score"),
        F.col("segment").cast("string").alias("segment"),
        F.col("recommended_channel_primary").cast("string").alias("recommended_channel_primary"),
        F.col("recommended_channel").cast("string").alias("recommended_channel"),
        F.col("recommended_offer").cast("string").alias("recommended_offer"),
        F.col("next_best_action").cast("string").alias("next_best_action"),
        F.col("mobile_app_user").cast("boolean").alias("mobile_app_user"),
        F.col("onboarding_mode").cast("string").alias("onboarding_mode"),
        F.col("consent_any_marketing").cast("boolean").alias("consent_any_marketing"),
        F.col("consent_push").cast("boolean").alias("consent_push"),
        F.col("consent_email").cast("boolean").alias("consent_email"),
        F.col("contact_priority").cast("int").alias("contact_priority"),
        F.col("score_date").cast("date").alias("score_date"),
    )
    targeting_sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGETING_TABLE)

    metrics_df = pd.DataFrame(
        [
            {"metric": "roc_auc", "value": metrics["roc_auc"], "model_version": metrics["model_version"]},
            {"metric": "average_precision", "value": metrics["average_precision"], "model_version": metrics["model_version"]},
            {"metric": "high_threshold", "value": metrics["high_threshold"], "model_version": metrics["model_version"]},
            {"metric": "medium_threshold", "value": metrics["medium_threshold"], "model_version": metrics["model_version"]},
            {"metric": "training_rows", "value": metrics["training_rows"], "model_version": metrics["model_version"]},
            {"metric": "positive_rate", "value": metrics["positive_rate"], "model_version": metrics["model_version"]},
        ]
    )
    spark.createDataFrame(metrics_df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(METRICS_TABLE)


training_df = load_training_frame()
scoring_df = load_scoring_frame()
model, metrics = fit_model(training_df)
state_df = build_state_table(scoring_df, model, metrics)
targeting_df = build_targeting_table(state_df)
segment_summary_df = build_segment_summary(state_df)
persist_tables(state_df, targeting_df, metrics)

print(json.dumps(metrics, indent=2))
print(json.dumps(segment_summary_df.to_dict(orient="records"), indent=2))
display(spark.table(METRICS_TABLE))
display(spark.createDataFrame(segment_summary_df))
display(spark.table(STATE_TABLE).limit(10))
display(spark.table(TARGETING_TABLE).limit(10))


{
  "roc_auc": 0.8958827586717429,
  "average_precision": 0.14936953745647003,
  "high_threshold": 0.830583716231821,
  "medium_threshold": 0.22104170663870756,
  "training_rows": 10000,
  "positive_rate": 0.0321,
  "model_version": "logreg_v3_databricks_uc_v5"
}
[
  {
    "segment": "LOW",
    "client_count": 5857
  },
  {
    "segment": "MEDIUM",
    "client_count": 3395
  },
  {
    "segment": "HIGH",
    "client_count": 748
  }
]


metric,value,model_version
roc_auc,0.8958827586717429,logreg_v3_databricks_uc_v5
average_precision,0.14936953745647003,logreg_v3_databricks_uc_v5
high_threshold,0.830583716231821,logreg_v3_databricks_uc_v5
medium_threshold,0.22104170663870756,logreg_v3_databricks_uc_v5
training_rows,10000.0,logreg_v3_databricks_uc_v5
positive_rate,0.0321,logreg_v3_databricks_uc_v5


segment,client_count
LOW,5857
MEDIUM,3395
HIGH,748


record_id,business_date,region,client_id,snapshot_date,propensity_score,segment,fuel_spend_30d,fuel_txn_count_30d,fuel_spend_90d,service_txn_count_90d,highway_vignette_flag,insurance_payment_flag,days_since_last_insurance,avg_txn_amount,total_txn_count_90d,age_band,dq_flag
077cabf6-2415-4f63-8bc2-7a50f42e4e64,2025-12-31,OLOMOUCKÝ KRAJ,CLI_E_00538,2025-12-31,0.1007148630389227,LOW,0.0,0,0.0,0,0,0,null,2248.809285714286,14,55+,PASS
bbdd2a48-b249-42aa-825d-00f5f893f81c,2025-12-31,KRÁLOVÉHRADECKÝ KRAJ,CLI_E_03182,2025-12-31,0.23072769641074928,MEDIUM,0.0,0,0.0,0,0,0,null,1508.2329411764706,17,35-44,PASS
ff33f9c9-36e7-400a-941a-a7877d8e3a96,2025-12-31,SCHLESWIG-HOLSTEIN,CLI_E_05766,2025-12-31,0.23334972645943777,MEDIUM,0.0,0,0.0,0,0,0,null,2306.0310526315793,19,25-34,PASS
8dec941f-a32a-409f-bd76-0099b0ae593e,2025-12-31,LIBERECKÝ KRAJ,CLI_E_00292,2025-12-31,0.030700479708199366,LOW,0.0,0,0.0,0,0,0,null,1360.5247826086954,23,18-24,PASS
58157136-5fa9-425c-804f-3a72d1d994f3,2025-12-31,PLZEŇSKÝ KRAJ,CLI_E_00358,2025-12-31,0.7767292099816009,MEDIUM,1331.08,1,2360.7200000000003,2,0,0,null,2116.7313333333336,15,25-34,PASS
04d4fdc0-dd28-4eeb-933b-3abd5470940c,2025-12-31,RHEINLAND-PFALZ,CLI_E_03378,2025-12-31,0.23197905211430303,MEDIUM,0.0,0,0.0,0,0,0,null,1620.430909090909,22,35-44,PASS
c9aabf54-7463-4beb-815e-4fdfec565beb,2025-12-31,MORAVSKOSLEZSKÝ KRAJ,CLI_E_02736,2025-12-31,0.1954234972470128,LOW,0.0,0,0.0,0,0,0,null,1445.429285714286,14,25-34,PASS
b60000ab-04fe-47da-bf43-5d5b6f113c7e,2025-12-31,MORAVSKOSLEZSKÝ KRAJ,CLI_E_08744,2025-12-31,0.21918029982822565,LOW,0.0,0,0.0,0,0,0,null,1993.6435294117648,17,25-34,PASS
8c9ef41a-f461-4ca9-932c-b1ab944027af,2025-12-31,JIHOMORAVSKÝ KRAJ,CLI_E_03615,2025-12-31,0.036389464304410854,LOW,0.0,0,0.0,0,0,0,null,1821.6746666666668,15,18-24,PASS
3e67610d-6122-4286-8d3f-800d54ae971a,2025-12-31,NIEDERSACHSEN,CLI_E_06205,2025-12-31,0.34032340424149804,MEDIUM,0.0,0,1106.82,0,0,0,null,2098.97,11,35-44,PASS


client_id,propensity_score,segment,recommended_channel_primary,recommended_channel,recommended_offer,next_best_action,mobile_app_user,onboarding_mode,consent_any_marketing,consent_push,consent_email,contact_priority,score_date
CLI_E_05038,0.17999970371623336,LOW,banner_in_ib,banner_in_ib,vehicle_insurance_offer,contact_client,true,pure_digital,true,false,false,6251,2025-12-31
CLI_E_07726,0.17998110781432614,LOW,banner_in_ib,banner_in_ib,vehicle_insurance_offer,contact_client,true,assisted,true,false,false,6252,2025-12-31
CLI_E_02221,0.17996822535289275,LOW,banner_in_ib,banner_in_ib,vehicle_insurance_offer,contact_client,true,assisted,true,false,false,6253,2025-12-31
CLI_E_08673,0.1799500110811057,LOW,banner_in_ib,banner_in_ib,vehicle_insurance_offer,contact_client,false,assisted,true,false,false,6254,2025-12-31
CLI_E_08665,0.1798725948622542,LOW,null,null,vehicle_insurance_offer,no_marketing_consent,false,assisted,false,false,false,6255,2025-12-31
CLI_E_07391,0.1798249401308912,LOW,banner_in_ib,banner_in_ib,vehicle_insurance_offer,contact_client,false,assisted,true,false,false,6256,2025-12-31
CLI_E_09337,0.17979213826714788,LOW,null,null,vehicle_insurance_offer,no_marketing_consent,false,pure_digital,false,false,false,6257,2025-12-31
CLI_E_01892,0.17976144848720627,LOW,null,null,vehicle_insurance_offer,no_marketing_consent,true,pure_digital,false,false,false,6258,2025-12-31
CLI_E_03740,0.17969785006355216,LOW,null,null,vehicle_insurance_offer,no_marketing_consent,true,pure_digital,false,false,false,6259,2025-12-31
CLI_E_08370,0.1796356970192432,LOW,null,null,vehicle_insurance_offer,no_marketing_consent,true,assisted,false,false,false,6260,2025-12-31
